# Extract Boundary Tokens - IMPROVED VERSION
Records ABSOLUTE corpus positions, not chunk-relative offsets.
This simplifies post-processing dramatically - no chunk position estimation needed!

In [ ]:
from google.colab import drive
import os, time
drive.mount('/content/drive')
time.sleep(2)
os.chdir('/content/drive/MyDrive/khabar-segmentation')
print(f"Working directory: {os.getcwd()}")

In [ ]:
!pip install transformers torch tqdm scipy -q
print("Dependencies OK")

In [ ]:
import json
import torch
import numpy as np
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForTokenClassification

# Load model from Drive
model_path = Path('checkpoints/camelbert_binary_classification_final')
print(f"Loading model from {model_path}...")
tokenizer = AutoTokenizer.from_pretrained(str(model_path))
model = AutoModelForTokenClassification.from_pretrained(str(model_path))
model.eval()
if torch.cuda.is_available():
    model = model.cuda()
print("Model loaded")

In [ ]:
# ===== CONFIGURATION =====
# Change this to process different texts
CORPUS_NAME = 'alDarrab'  # or 'ibjawzi', etc.
corpus_file = Path(f'data/processed/{CORPUS_NAME}_clean.txt')

print(f"Loading corpus: {corpus_file}")
with open(corpus_file, encoding='utf-8') as f:
    text = f.read()
print(f"Corpus: {len(text):,} chars, {len(text.split()):,} words")

In [ ]:
# Process FULL TEXT in overlapping chunks
# KEY IMPROVEMENT: Track absolute positions, not chunk-relative
from tqdm import tqdm
from scipy.special import softmax

print("Running inference on FULL CORPUS...")
print(f"Text length: {len(text):,} chars\n")

all_tokens = []
all_predictions = []
all_offsets = []  # ABSOLUTE corpus positions
all_probabilities = []

CHUNK_SIZE = 500
OVERLAP = 50
chunks_processed = 0

# Split text into overlapping chunks
for start_char in tqdm(range(0, len(text), CHUNK_SIZE), desc="Processing chunks"):
    end_char = min(start_char + CHUNK_SIZE + OVERLAP, len(text))
    chunk_text = text[start_char:end_char]

    # Encode chunk
    encoded = tokenizer(
        chunk_text,
        return_tensors='pt',
        return_offsets_mapping=True,
        truncation=False,
        padding=False,
    )

    # Run inference
    with torch.no_grad():
        if torch.cuda.is_available():
            outputs = model(
                input_ids=encoded['input_ids'].cuda(),
                attention_mask=encoded['attention_mask'].cuda()
            )
        else:
            outputs = model(**encoded)
        logits = outputs.logits[0]

    # Get predictions and probabilities
    preds = np.argmax(logits.cpu().numpy(), axis=-1)
    probs = softmax(logits.cpu().numpy(), axis=-1)
    boundary_probs = probs[:, 1].tolist()  # Probability of boundary class
    
    tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])
    offsets_chunk_relative = encoded['offset_mapping'][0].numpy()

    # IMPROVEMENT: Convert chunk-relative offsets to absolute corpus positions
    # FIX: Convert numpy int64 to Python int for JSON serialization
    offsets_absolute = [[int(start_char + int(offset[0])), int(start_char + int(offset[1]))]
                       for offset in offsets_chunk_relative]

    # Avoid duplicate tokens from overlap
    if chunks_processed > 0 and len(all_tokens) > 0:
        # Skip first 5 tokens of new chunk to avoid duplicates
        skip = 5
        preds = preds[skip:]
        tokens = tokens[skip:]
        offsets_absolute = offsets_absolute[skip:]
        boundary_probs = boundary_probs[skip:]

    all_tokens.extend(tokens)
    all_predictions.extend(preds.tolist())
    all_offsets.extend(offsets_absolute)
    all_probabilities.extend(boundary_probs)

    chunks_processed += 1

print(f"\nProcessing complete")
print(f"  Chunks processed: {chunks_processed}")
print(f"  Total tokens: {len(all_tokens):,}")
print(f"  Boundary tokens: {sum(all_predictions):,}")
print(f"  Percentage: {100 * sum(all_predictions) / len(all_predictions):.2f}%")

In [ ]:
# Validate offset positions
print("=== VALIDATION: Offset Accuracy ===")

sample_size = min(100, len(all_tokens))
matches = 0

for i in range(sample_size):
    token = all_tokens[i]
    offset = all_offsets[i]
    char_start, char_end = offset
    
    # Check if token matches corpus text at this position
    if 0 <= char_start < len(text) and 0 <= char_end <= len(text):
        corpus_text = text[char_start:char_end]
        # For subword tokens (##...), remove prefix for comparison
        token_clean = token.lstrip('#')
        if corpus_text == token_clean:
            matches += 1

accuracy = 100 * matches / sample_size
print(f"Offset accuracy: {matches}/{sample_size} = {accuracy:.1f}%")
print(f"\n✓ Offsets are ABSOLUTE corpus positions (not chunk-relative)")
print(f"✓ Post-processing can use offsets directly without chunk estimation")

In [ ]:
# Save raw inference with ABSOLUTE offsets
print("\nSaving raw inference to JSON...")

# Verify all lists have same length
assert len(all_tokens) == len(all_predictions) == len(all_offsets) == len(all_probabilities), \
    f"Length mismatch: tokens={len(all_tokens)}, preds={len(all_predictions)}, offsets={len(all_offsets)}, probs={len(all_probabilities)}"

# Ensure all values are Python native types (not numpy)
results = {
    'metadata': {
        'corpus': f'{CORPUS_NAME}_clean.txt',
        'corpus_size_chars': int(len(text)),
        'total_tokens': int(len(all_tokens)),
        'model': 'camelbert_binary_classification_final',
        'processing_method': 'Chunked inference with ABSOLUTE offset tracking',
        'chunk_size': int(CHUNK_SIZE),
        'overlap': int(OVERLAP),
        'chunks_processed': int(chunks_processed),
        'boundary_tokens_count': int(sum(all_predictions)),
        'boundary_percentage': float(round(100 * sum(all_predictions) / len(all_tokens), 2)),
        'offset_format': 'ABSOLUTE corpus character positions [char_start, char_end]',
        'note': 'No chunk position estimation needed in post-processing!',
    },
    'inference_results': {
        'total_tokens': int(len(all_tokens)),
        'tokens': all_tokens,
        'offsets': all_offsets,  # [[start, end], ...] - ABSOLUTE positions (already int)
        'predictions': [int(p) for p in all_predictions],  # [0, 1, 0, 1, ...]
        'probabilities': [float(p) for p in all_probabilities],  # [0.05, 0.98, 0.02, 0.99, ...]
    }
}

output_file = Path(f'results/{CORPUS_NAME}/camelbert_{CORPUS_NAME}_raw_inference_improved.json')
output_file.parent.mkdir(parents=True, exist_ok=True)
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

file_size = output_file.stat().st_size / (1024 * 1024)
print(f"✓ Saved: {output_file}")
print(f"  Size: {file_size:.2f} MB")
print(f"\nInference Summary:")
print(f"  Corpus: {len(text):,} chars")
print(f"  Total tokens: {len(all_tokens):,}")
print(f"  Boundary tokens: {sum(all_predictions):,}")
print(f"  Percentage: {100 * sum(all_predictions) / len(all_predictions):.2f}%")
print(f"  Mean boundary probability: {np.mean(all_probabilities):.4f}")
print(f"\nNext Step:")
print(f"  1. Download JSON to local machine")
print(f"  2. Run: python scripts/convert_boundary_tokens_direct.py \\")
print(f"           --raw_inference {output_file} \\")
print(f"           --corpus data/processed/{CORPUS_NAME}_clean.txt \\")
print(f"           --output results/{CORPUS_NAME}/camelbert_{CORPUS_NAME}_char_boundaries.json")
print(f"\n✓ Done!")

In [ ]:
# Validate saved format
print("=== VALIDATION: Raw Inference Format ===")

with open(output_file, 'r', encoding='utf-8') as f:
    saved_data = json.load(f)

inf = saved_data['inference_results']
meta = saved_data['metadata']

print(f"Offset Format: {meta['offset_format']}")
print(f"\nFormat Check:")
print(f"  'tokens': {len(inf['tokens'])} items ✓")
print(f"  'offsets': {len(inf['offsets'])} items ✓")
print(f"  'predictions': {len(inf['predictions'])} items ✓")
print(f"  'probabilities': {len(inf['probabilities'])} items ✓")

print(f"\nSample Data (first 5 tokens):")
for i in range(min(5, len(inf['tokens']))):
    tok = inf['tokens'][i]
    off = inf['offsets'][i]
    pred = inf['predictions'][i]
    prob = inf['probabilities'][i]
    # Show text at this position
    try:
        text_snippet = text[off[0]:min(off[1], off[0]+20)]
        print(f"  [{i}] {tok:15s} offset={off} text='{text_snippet}' pred={pred} prob={prob:.4f}")
    except:
        print(f"  [{i}] {tok:15s} offset={off} pred={pred} prob={prob:.4f}")

print(f"\nBoundary Token Samples (pred=1):")
boundary_indices = [i for i, p in enumerate(inf['predictions']) if p == 1][:5]
for i in boundary_indices:
    tok = inf['tokens'][i]
    off = inf['offsets'][i]
    prob = inf['probabilities'][i]
    try:
        text_snippet = text[off[0]:min(off[1], off[0]+20)]
        print(f"  [{i}] {tok:15s} offset={off} text='{text_snippet}' prob={prob:.4f}")
    except:
        print(f"  [{i}] {tok:15s} offset={off} prob={prob:.4f}")

print(f"\n✓ Format validated and ready for post-processing")
print(f"✓ You can now use convert_boundary_tokens_direct.py directly!")